# 04 — Run T and VT

For each of the first `N_QUESTIONS` questions (in the fixed order from notebook 03), score A–D twice:
- **T**: transcript + question + options;
- **VT**: the same text + 64 silent video frames.

Each result is appended to `outputs/results.jsonl` as soon as it is computed. Questions already in the file are skipped,
so you can stop at any time, or raise `N_QUESTIONS` and rerun — only new questions are computed.

No answer keys are used here; accuracy and comparisons are done later.

In [ ]:
N_QUESTIONS = 3

In [ ]:
import sys, json, time, tomllib
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import qwen_omni as qo

CFG = tomllib.loads((ROOT / "configs/pilot.toml").read_text())
OUT = ROOT / "outputs/results.jsonl"
OUT.parent.mkdir(exist_ok=True)

questions = pd.read_parquet(ROOT / "data/processed/emotion_questions.parquet").head(N_QUESTIONS)
done = {(r["question_id"], r["condition"]) for r in map(json.loads, OUT.open())} if OUT.exists() else set()
print(f"{len(questions)} questions selected, {len(done)} results already in {OUT.name}")

## Load the model (~18 GB RAM, a few minutes)

In [ ]:
processor, model = qo.load(CFG["model_revision"])

## Show the exact prompt for the first question

In [ ]:
first = questions.iloc[0]
print(qo.make_inputs(processor, qo.SYSTEM_PROMPT, qo.build_prompt(first))[1])

## Run

In [ ]:
for q in questions.itertuples():
    user_text = qo.build_prompt(q._asdict())
    for condition in ["T", "VT"]:
        if (q.question_id, condition) in done:
            continue
        t0 = time.time()
        if condition == "T":
            inputs, _ = qo.make_inputs(processor, qo.SYSTEM_PROMPT, user_text)
        else:
            video = ROOT / "data/raw/all_videos" / f"{q.movie_title}.mp4"
            frames, fps = qo.sample_frames(video, q.t_i, q.t_j, q.n_frames, CFG["max_pixels"])
            inputs, _ = qo.make_inputs(processor, qo.SYSTEM_PROMPT, user_text, frames, fps)
        r = qo.score_options(model, processor, inputs)
        row = {"order": q.order, "question_id": q.question_id, "condition": condition,
               **{f"prob_{L}": p for L, p in zip(qo.LETTERS, r["prob"])},
               "mass_on_letters": r["mass_on_letters"], "n_input_tokens": r["n_input_tokens"],
               "seconds": round(time.time() - t0, 1), "model_revision": CFG["model_revision"],
               "n_frames": q.n_frames if condition == "VT" else 0, "max_pixels": CFG["max_pixels"]}
        with OUT.open("a") as f:
            f.write(json.dumps(row) + "\n")
        print(f"{q.order:4d} {q.question_id} {condition:2s} {r['n_input_tokens']:5d} tokens  {row['seconds']:5.0f} s  "
              f"probs {[round(p, 2) for p in r['prob']]}")

## Results so far

In [ ]:
results = pd.read_json(OUT, lines=True)
results.pivot(index=["order", "question_id"], columns="condition", values=["prob_A", "prob_B", "prob_C", "prob_D", "seconds"]).round(3)